# Koala-Kangaroo YOLO

## Logging And Setup

We setup our logging using `wandb`
We also initialize our global variables

In [ ]:
from ultralytics import settings
import os
os.environ["WANDB_DISABLE_SSL"] = "true"
settings.update({"wandb": True,
                 "clearml": False,
                 "comet": False})
YOLO_MODEL = "yolo11s.pt"
PROJECT_NAME = "koala-kangaroo"
EXPERIMENT_NAME = "exp1.3"
EXPERIMENT_CONFIG = f"train_config_{EXPERIMENT_NAME}.yaml"
TEST_VIDEO_PATH = "test_video.mp4"
TEST_IMAGE_PATH = ["test_image.jpg","image_0010_together.jpg", "image_0011_together.jpg", "image_0012_together.jpg"]
DATASET_PATH = "datasets/yolo_koala_kangaroo.v1-original.yolov11/data.yaml"
BEST_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt"
SAVED_MODEL_PATH = f"{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best_int8_openvino_model" #koala-kangaroo/exp1/weights/best_int8_openvino_model/


## Training

We specify the training path for our dataset.

For batch size, given we are training in our laptop and faced GPU out of memory problem, we leave it to Yolo to auto batch using -1 which seemed to solve the memory problem.


In [14]:
!ls -la datasets/yolo_koala_kangaroo.v1-original.yolov11/train/images | wc -l

242


In [15]:
from ultralytics import YOLO
from ultralytics import settings

model = YOLO(YOLO_MODEL)  # Load a pre-trained YOLO model
result = model.train(data=DATASET_PATH,
                     save_period=1, # save every epoch
                     batch=-1, # auto batch size, previously set to 16,64 and caused us to crash for GPU memory reasons
                     device=0, # use GPU 0
                     project=PROJECT_NAME, # set project name  for logging in wandb
                     name=EXPERIMENT_NAME, # set experiment name for logging in wandb
                     cfg=EXPERIMENT_CONFIG, # use yolov11n config
                     plots=True)

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 3769MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=train_config_exp1.2.yaml, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo_koala_kangaroo.v1-original.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=exp1.2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics

lr/pg0,▃▆███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
lr/pg1,▃▆███▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁
lr/pg2,██▇▇▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁
metrics/mAP50(B),▂▁▂▂▄▆▇▇▇▇▇▇▇▇▇▇▇██▇██▇▇▇██▇█▇██████████
metrics/mAP50-95(B),▂▁▁▂▃▃▅▅▅▅▆▆▅▆▇▆▆▆▆▆▇▇▇▇▇▇▇▇█▇▇█▇▇██████
metrics/precision(B),▁▃▃▄▂▄▄▄▄▆▆▆▆▆▆▆▇▆▆▇▇▇▇▇▇██▇▇▇▇▇▇▇▇▇█▇▇█
metrics/recall(B),▁▁▄▁▄▄▆▆▇▅▆▆▇▆▇▇▇█▆▇▇▇█▇▇▆▆▇▆▇▇▇▇▇█████▇
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


## Validation

Here we are looking for mAP50 and mAP50-95 score

In [16]:
model = YOLO(BEST_MODEL_PATH)
metrics = model.val(data=DATASET_PATH, device="0")

# Retrieve precision and recall
precision = metrics.box.mp  # Mean Precision
recall = metrics.box.mr     # Mean Recall

# Calculate the F1 score
print(metrics)
if (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
    print(f"F1 Score: {f1_score}")
else:
    print("Precision and recall are zero, cannot calculate F1 score.")
print(f"Precision: {precision}, Recall: {recall}")
print("Validation complete.")

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 2050, 3769MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1401.5±545.4 MB/s, size: 47.5 KB)
val: Scanning /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/datasets/yolo_koala_kangaroo.v1-original.yolov11/valid/labels.cache... 66 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 179.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.5it/s 2.0s0.5ss
                   all         66         95      0.925      0.681      0.793      0.456
              Kangaroo         31         54        0.9      0.704      0.768      0.471
                 Koala         36         41       0.95      0.659      0.819      0.441
Speed: 3.9ms preprocess, 19.4ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /ho

## Export

In [17]:
model = YOLO(BEST_MODEL_PATH)
exported_path = model.export(format="openvino", int8=True)

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.1+cu128 CPU (12th Gen Intel Core i5-12450HX)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from 'koala-kangaroo/exp1.2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (18.3 MB)

OpenVINO: starting export with openvino 2025.4.1-20426-82bbf0292c5-releases/2025/4...
OpenVINO: collecting INT8 calibration images from 'data=coco8.yaml'
Fast image access ✅ (ping: 0.0±0.0 ms, read: 345.5±136.9 MB/s, size: 54.0 KB)
Scanning /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 8.5Kit/s 0.0s0s
WARNING ⚠️ OpenVINO: >300 images recommended for INT8 calibration, found 4 images.
INFO:nncf:15 ignored nodes were found by patterns in the NNCFGraph
I

Output()

Output()

OpenVINO: export success ✅ 10.5s, saved as 'koala-kangaroo/exp1.2/weights/best_int8_openvino_model/' (9.8 MB)

Export complete (10.9s)
Results saved to /home/ray/Projects/25S2-C-NYP-ITI121---Applied-Deep-Learning-Assignment2/koala-kangaroo/exp1.2/weights
Predict:         yolo predict task=detect model=koala-kangaroo/exp1.2/weights/best_int8_openvino_model imgsz=640 int8 
Validate:        yolo val task=detect model=koala-kangaroo/exp1.2/weights/best_int8_openvino_model imgsz=640 data=datasets/yolo_koala_kangaroo.v1-original.yolov11/data.yaml int8 
Visualize:       https://netron.app


## Inference

In [18]:
import ultralytics
from ultralytics import YOLO
from PIL import Image

source = TEST_IMAGE_PATH
model = YOLO(BEST_MODEL_PATH, task='detect')
result = model(source, conf=0.5, iou=0.6)

# Visualize the results
for i, r in enumerate(result):
    print(r)
    # Plot results image
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = Image.fromarray(im_bgr[..., ::-1])  # RGB-order PIL image

    # Show results to screen (in supported environments)
    r.show()

    # Save results to disk
    r.save(filename=f"results-{EXPERIMENT_NAME}-{i}.jpg")


0: 640x640 2 Kangaroos, 1 Koala, 15.2ms
1: 640x640 (no detections), 15.2ms
2: 640x640 1 Koala, 15.2ms
3: 640x640 2 Kangaroos, 1 Koala, 15.2ms
Speed: 1.6ms preprocess, 15.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'Kangaroo', 1: 'Koala'}
obb: None
orig_img: array([[[ 88, 231, 192],
        [ 83, 215, 178],
        [ 94, 205, 173],
        ...,
        [ 95, 130, 116],
        [ 96, 131, 117],
        [ 97, 132, 118]],

       [[ 77, 216, 178],
        [ 73, 204, 167],
        [ 87, 198, 166],
        ...,
        [ 92, 127, 113],
        [ 93, 128, 114],
        [ 94, 129, 115]],

       [[ 68, 200, 163],
        [ 69, 193, 157],
        [ 82, 193, 161],
        ...,
        [ 89, 124, 110],
        [ 89, 124, 110],
        [ 89, 124, 110]],

       ...,

       [[ 56,  66,  66],
        [ 60,  71,  69],
        [ 61


(gthumb:103593): Gtk-WARNING **: 15:51:07.750: Getting screensaver status failed: GDBus.Error:org.freedesktop.DBus.Error.ServiceUnknown: The name org.gnome.Shell.ScreenShield was not provided by any .service files


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'Kangaroo', 1: 'Koala'}
obb: None
orig_img: array([[[  1,   1,   1],
        [  1,   1,   1],
        [  1,   1,   1],
        ...,
        [  6,   9,  13],
        [  6,   9,  13],
        [  6,   9,  13]],

       [[  1,   1,   1],
        [  1,   1,   1],
        [  1,   1,   1],
        ...,
        [  6,   9,  13],
        [  6,   9,  13],
        [  6,   9,  13]],

       [[  1,   1,   1],
        [  1,   1,   1],
        [  1,   1,   1],
        ...,
        [  6,   9,  13],
        [  6,   9,  13],
        [  6,   9,  13]],

       ...,

       [[ 43, 161,  19],
        [ 43, 160,  21],
        [ 57, 169,  38],
        ...,
        [163, 198, 231],
        [160, 193, 226],
        [158, 193, 226]],

       [[ 41, 153,  11],
        [ 48, 160,  19],
        [ 65, 174,  42],
        ...,
        [146, 183, 217],
        [146, 181, 215],

## Video

In [24]:
from ultralytics import YOLO
import cv2
# Load the YOLO model
model = YOLO(SAVED_MODEL_PATH, task="detect")

# Open the video file
video_path = TEST_VIDEO_PATH
cap = cv2.VideoCapture(video_path)

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the frame on GPU Device 0
        results = model(frame, device="cpu")

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        # Display the annotated frame
        cv2.imshow("YOLO Inference", annotated_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

Loading koala-kangaroo/exp1.2/weights/best_int8_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...

0: 640x640 (no detections), 30.9ms
Speed: 2.1ms preprocess, 30.9ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 32.7ms
Speed: 1.1ms preprocess, 32.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 31.3ms
Speed: 1.2ms preprocess, 31.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 29.7ms
Speed: 1.3ms preprocess, 29.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 33.5ms
Speed: 1.2ms preprocess, 33.5ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 29.2ms
Speed: 1.4ms preprocess, 29.2ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 29.2ms
Speed: 1.5ms prep

## Write Video

In [24]:
from ultralytics import YOLO
import cv2
# from tqdm import tqdm
from tqdm.auto import tqdm

def write_video(video_in_filepath, video_out_filepath, model):
    # Open the video file

    video_reader = cv2.VideoCapture(video_in_filepath)

    nb_frames = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_h = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_w = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    fps = video_reader.get(cv2.CAP_PROP_FPS)

    video_writer = cv2.VideoWriter(video_out_filepath,
                            cv2.VideoWriter_fourcc(*'mp4v'),
                            fps,
                            (frame_w, frame_h))

    # Loop through the video frames
    for i in tqdm(range(nb_frames)):
        # Read a frame from the video
        success, frame = video_reader.read()

        if success:
            # Run YOLO inference on the frame on GPU Device 0
            results = model(frame, conf=0.6, device=0)

            # Visualize the results on the frame
            annotated_frame = results[0].plot()

            # Write the annotated frame
            video_writer.write(annotated_frame)

    video_reader.release()
    video_writer.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

In [25]:
from pathlib import Path
import os

video_in_file = TEST_VIDEO_PATH
basename = Path(video_in_file).stem
video_out_file = os.path.join(basename + '_detected' + '.mp4')
model = YOLO(SAVED_MODEL_PATH, task="detect")
write_video(video_in_file, video_out_file, model)

  0%|          | 0/1500 [00:00<?, ?it/s]

Loading koala-kangaroo/exp1/weights/best_int8_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...

0: 640x640 (no detections), 32.5ms
Speed: 2.1ms preprocess, 32.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 44.4ms
Speed: 1.9ms preprocess, 44.4ms inference, 8.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 35.4ms
Speed: 3.2ms preprocess, 35.4ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 36.3ms
Speed: 2.6ms preprocess, 36.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 38.6ms
Speed: 1.8ms preprocess, 38.6ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 38.4ms
Speed: 1.8ms preprocess, 38.4ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 32.6ms
Speed: 10.3ms prepr